# 03b — Graph construction for full dataset

Construct B-B, C-C, and B-C graphs using full single-cell cohort artifacts.


## Install dependencies


In [13]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torch_geometric', 'numpy', 'scipy', 'scikit-learn'])


0

## Setup


In [14]:
from pathlib import Path
import os, sys, json, time, random
import numpy as np
import pandas as pd

USE_DRIVE = True
SEED = 42

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/BulkCellGNN_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, '/content/drive/MyDrive/BulkCellGNN_data')

random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)

print('DATA_DIR =', DATA_DIR)

import torch
from sklearn.decomposition import PCA
from sklearn.neighbors import kneighbors_graph
import bulkcell_gnn as bcg

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR = /content/drive/MyDrive/BulkCellGNN_data
device = cpu


## Load arrays


In [15]:
bulk_expr_sym = np.load(DATA_DIR / 'bulk_expr_sym.npy')
bulk_genes_sym = np.load(DATA_DIR / 'bulk_genes_sym.npy', allow_pickle=True)
cell_expr_full = np.load(DATA_DIR / 'cell_expr_full.npy')
cell_genes_full = np.load(DATA_DIR / 'cell_genes_full.npy', allow_pickle=True)
print('bulk_expr_sym:', bulk_expr_sym.shape)
print('cell_expr_full:', cell_expr_full.shape)


bulk_expr_sym: (536, 22880)
cell_expr_full: (47285, 5000)


## Shared gene alignment


In [16]:
bg = {str(g): i for i, g in enumerate(bulk_genes_sym)}
cg = {str(g): i for i, g in enumerate(cell_genes_full)}
shared = sorted(set(bg.keys()) & set(cg.keys()))
print('Shared genes:', len(shared))
if len(shared) < 500:
    print('WARNING: shared genes below 500')

idx_b = [bg[g] for g in shared]
idx_c = [cg[g] for g in shared]
bulk_shared_full = bulk_expr_sym[:, idx_b].astype(np.float32)
cell_shared_full = cell_expr_full[:, idx_c].astype(np.float32)
print('bulk_shared_full:', bulk_shared_full.shape)
print('cell_shared_full:', cell_shared_full.shape)


Shared genes: 3263
bulk_shared_full: (536, 3263)
cell_shared_full: (47285, 3263)


## Build B-B edges with degree target 5-30


In [17]:
t0 = time.time()
threshold = 0.70
for _ in range(20):
    edge_BB_full, sim = bcg.build_bulk_graph(bulk_shared_full, gsva_scores=None, threshold=threshold)
    adj = (sim >= threshold).float()
    mean_deg = float(adj.sum().item()) / (2 * adj.shape[0])
    print(f'threshold={threshold:.3f} mean_deg={mean_deg:.2f} |E_BB|={edge_BB_full.shape[1]}')
    if 5 <= mean_deg <= 30:
        break
    threshold = threshold - 0.03 if mean_deg < 5 else threshold + 0.03
    threshold = float(max(0.20, min(0.99, threshold)))

torch.save(edge_BB_full, DATA_DIR / 'edge_BB_full.pt')
print('Saved edge_BB_full.pt | elapsed_sec=', round(time.time() - t0, 2))


threshold=0.700 mean_deg=0.03 |E_BB|=28
threshold=0.670 mean_deg=0.06 |E_BB|=64
threshold=0.640 mean_deg=0.12 |E_BB|=128
threshold=0.610 mean_deg=0.18 |E_BB|=188
threshold=0.580 mean_deg=0.26 |E_BB|=274
threshold=0.550 mean_deg=0.34 |E_BB|=360
threshold=0.520 mean_deg=0.45 |E_BB|=482
threshold=0.490 mean_deg=0.67 |E_BB|=716
threshold=0.460 mean_deg=0.93 |E_BB|=1000
threshold=0.430 mean_deg=1.40 |E_BB|=1502
threshold=0.400 mean_deg=2.08 |E_BB|=2232
threshold=0.370 mean_deg=3.15 |E_BB|=3382
threshold=0.340 mean_deg=4.78 |E_BB|=5128
threshold=0.310 mean_deg=7.03 |E_BB|=7532
Saved edge_BB_full.pt | elapsed_sec= 0.78


## Build C-C edges (kNN=15 on PCA)


In [18]:
t0 = time.time()
n_components = min(50, cell_shared_full.shape[1] - 1, cell_shared_full.shape[0] - 1)
Z = PCA(n_components=n_components, random_state=SEED).fit_transform(cell_shared_full)
A = kneighbors_graph(Z, n_neighbors=15, mode='connectivity', include_self=False)
A = A.maximum(A.T)
try:
    from torch_geometric.utils import from_scipy_sparse_matrix
except ImportError:
    from torch_geometric.utils import from_scipy_sparse_array as from_scipy_sparse_matrix
edge_CC_full, _ = from_scipy_sparse_matrix(A)
torch.save(edge_CC_full, DATA_DIR / 'edge_CC_full.pt')
print('Saved edge_CC_full.pt:', edge_CC_full.shape, '| elapsed_sec=', round(time.time() - t0, 2))


Saved edge_CC_full.pt: torch.Size([2, 1107510]) | elapsed_sec= 27.02


## Build B-C edges with top-K=50 (batched)


In [19]:
t0 = time.time()
print('Building B-C with batch_size=50...')
edge_BC_full, w_bc_full = bcg.build_cross_modal_graph(bulk_shared_full, cell_shared_full, top_k=50, batch_size=50)
torch.save(edge_BC_full, DATA_DIR / 'edge_BC_full.pt')
torch.save(w_bc_full, DATA_DIR / 'edge_BC_full_weights.pt')
print('edge_BC_full:', edge_BC_full.shape)
print('weight range:', float(w_bc_full.min().item()), 'to', float(w_bc_full.max().item()))
print('elapsed_sec=', round(time.time() - t0, 2))


Building B-C with batch_size=50...
edge_BC_full: torch.Size([2, 26800])
weight range: -0.02401559427380562 to 0.39578935503959656
elapsed_sec= 10.66


## Save shared matrices and graph metadata


In [20]:
np.save(DATA_DIR / 'bulk_shared_full.npy', bulk_shared_full)
np.save(DATA_DIR / 'cell_shared_full.npy', cell_shared_full)
meta = {
    'n_bulk': int(bulk_shared_full.shape[0]),
    'n_cell': int(cell_shared_full.shape[0]),
    'n_shared_genes': int(bulk_shared_full.shape[1]),
    'bb_threshold': float(threshold),
    'bb_edges': int(edge_BB_full.shape[1]),
    'cc_edges': int(edge_CC_full.shape[1]),
    'bc_edges': int(edge_BC_full.shape[1]),
}
(DATA_DIR / 'graph_meta_full.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
print(json.dumps(meta, indent=2))


{
  "n_bulk": 536,
  "n_cell": 47285,
  "n_shared_genes": 3263,
  "bb_threshold": 0.3099999999999996,
  "bb_edges": 7532,
  "cc_edges": 1107510,
  "bc_edges": 26800
}


## Final verification


In [21]:
expected = ['edge_BB_full.pt','edge_CC_full.pt','edge_BC_full.pt','bulk_shared_full.npy','cell_shared_full.npy','graph_meta_full.json']
missing = []
for name in expected:
    p = DATA_DIR / name
    if p.exists():
        print('OK ', name)
    else:
        print('MISSING', name)
        missing.append(name)
if missing:
    raise FileNotFoundError('Missing outputs: ' + ', '.join(missing))
print('Notebook 03b complete')


OK  edge_BB_full.pt
OK  edge_CC_full.pt
OK  edge_BC_full.pt
OK  bulk_shared_full.npy
OK  cell_shared_full.npy
OK  graph_meta_full.json
Notebook 03b complete
